## 1. Setup & Parse Arguments

In [1]:
# Auto-reload modules
%load_ext autoreload
%autoreload 2

import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import argparse
import sys
from datetime import datetime

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Ti
GPU Memory: 17.18 GB


In [2]:
EXP_NAME = "experiment_1"

In [5]:
from config import (
    BATCH_SIZE, NUM_EPOCHS, LEARNING_RATE,
    MODE, FUSION_METHOD, FEATURE_MODE,
    CHECKPOINT_DIR, BEST_MODEL_NAME,
)

def parse_args():
    parser = argparse.ArgumentParser(description="Train Hybrid ECAPA-TDNN Model")
    
    # --- DATA PATHS (Chỉ cần chỉ định thư mục gốc) ---
    parser.add_argument("--base_dir", type=str, 
                        default=r"E:\speech_data\features\train\train_vi_7s",
                        help="Thư mục chứa các folder con: FBank, MFBE + Pitch...")
    parser.add_argument("--test_dir", type=str, 
                        default="C:/path/to/your/data/test",
                        help="Thư mục test")
    parser.add_argument("--output_dir", type=str, default="./outputs")
    
    # --- MODE & ARCHITECTURE ---
    parser.add_argument("--mode", type=int, choices=[1, 2, 3], default=MODE)
    parser.add_argument("--fusion_method", type=str, choices=["concat", "cross_attention", "gating"], default=FUSION_METHOD)
    parser.add_argument("--feature_mode", type=str, 
                        choices=["mfbe_pitch", "mfbe_only", "pitch_only"], default=FEATURE_MODE)
    
    # --- HYPERPARAMETERS ---
    parser.add_argument("--batch_size", type=int, default=64)
    parser.add_argument("--learning_rate", type=float, default=0.001)
    parser.add_argument("--num_epochs", type=int, default=100)
    parser.add_argument("--optimizer", type=str, choices=["adam", "sgd"], default="adam")
    parser.add_argument("--weight_decay", type=float, default=0.0001)
    
    # --- MISC ---
    parser.add_argument("--device", type=str, default="cuda")
    parser.add_argument("--mixed_precision", type=bool, default=True)
    parser.add_argument("--early_stop_patience", type=int, default=10)
    parser.add_argument("--lr_scheduler", type=str, default="plateau")
    parser.add_argument("--exp_name", type=str, default=None)
    parser.add_argument("--seed", type=int, default=42)
    
    args = parser.parse_args([])
    return args

args = parse_args()

print("="*60)
print("PARSED ARGUMENTS")
print("="*60)
print(f"  Mode: {args.mode} (1=PTM only, 2=Handcrafted only, 3=Fusion)")
print(f"  Fusion method: {args.fusion_method if args.mode == 3 else 'N/A'}")
print(f"  Feature mode: {args.feature_mode if args.mode in [2, 3] else 'N/A'}")
print(f"  Batch size: {args.batch_size}")
print(f"  Learning rate: {args.learning_rate}")
print(f"  Epochs: {args.num_epochs}")
print(f"  Exp name: {args.exp_name or 'Auto-generated'}")
print(f"  Seed: {args.seed}")
print("="*60)

PARSED ARGUMENTS
  Mode: 3 (1=PTM only, 2=Handcrafted only, 3=Fusion)
  Fusion method: concat
  Feature mode: mfbe_pitch
  Batch size: 64
  Learning rate: 0.001
  Epochs: 100
  Exp name: Auto-generated
  Seed: 42


## 2. Training

In [6]:
from train import train

# Generate experiment name if not provided
if args.exp_name is None:
    args.exp_name = f"mode{args.mode}_fusion_{args.fusion_method}_feat_{args.feature_mode}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

print("\n" + "="*80)
print("STARTING TRAINING")
print("="*80)
print(f"Experiment: {args.exp_name}\n")

model, history, exp_dir = train(args)

print(f"\n✓ Training completed!")
print(f"  Results saved to: {exp_dir}")
print("="*80)

ModuleNotFoundError: No module named 'torchinfo'

## 3. Training Curves

In [ ]:
# Load TensorBoard extension
%load_ext tensorboard

# Chạy giao diện TensorBoard trỏ vào thư mục chứa tất cả các experiments
%tensorboard --logdir ./outputs/experiments

## 4. Test Results

In [ ]:
import os
import json
from datetime import datetime
from train import load_checkpoint
from inference import evaluate_speaker_verification
from dataset import create_test_loader

# 1. Load Data
print("Loading Independent Test Data (Unseen Speakers)...")
test_loader, test_num_speakers = create_test_loader(
    test_dir=args.test_dir, 
    mode=args.mode, 
    feature_mode=args.feature_mode, 
    batch_size=args.batch_size, 
    num_workers=0
)
print(f"✓ Loaded {test_num_speakers} unseen speakers for testing")

# 2. Load Best Model từ exp_dir
best_model_path = os.path.join(exp_dir, "best_model.pth")
model, _, _, _ = load_checkpoint(best_model_path, model)
model = model.to(device)

# 3. Chạy Đánh giá EER & MinDCF
test_results = evaluate_speaker_verification(
    model=model, 
    data_loader=test_loader, 
    device=device,
    num_pairs=50000,
    p_target=0.05
)

# In kết quả ra màn hình
print("\n" + "="*50)
print("OPEN-SET BENCHMARK RESULTS (TEST SET)")
print("="*50)
for k, v in test_results.items():
    print(f"{k:<25}: {v:.4f}")
print("="*50)

# 4. TẠO FILE test_results.json VỚI TOÀN BỘ THÔNG SỐ
test_results_file = os.path.join(exp_dir, "test_results.json")

# Sử dụng vars(args) để lấy 100% các thông số cấu hình đã khai báo
final_test_data = {
    "test_timestamp": datetime.now().isoformat(),
    "config": vars(args),  # <-- Tự động lấy toàn bộ args (batch_size, lr, aam_margin, mode...)
    "metrics": test_results
}

with open(test_results_file, "w") as f:
    json.dump(final_test_data, f, indent=4)
    
print(f"✓ Đã lưu kết quả Test cùng TOÀN BỘ thông số cấu hình vào: {test_results_file}")

## 5. Gating Analysis (Only for Mode 3 + Gating)

In [ ]:
from train import analyze_gating_behavior
import matplotlib.image as mpimg

# Kiểm tra điều kiện để chạy phân tích
if args.mode == 3 and args.fusion_method == "gating":
    print("Đang thực hiện phân tích cơ chế Gating trên tập Test...")
    
    # Gọi hàm phân tích từ train.py
    # Lưu ý: Hàm này trả về 2 giá trị: gates (trọng số PTM) và labels
    gates, labels = analyze_gating_behavior(model, test_loader, device, exp_dir)
    
    # Hiển thị biểu đồ phân phối trọng số đã được lưu
    gate_plot_path = os.path.join(exp_dir, "gating_analysis", "gate_distribution.png")
    if os.path.exists(gate_plot_path):
        plt.figure(figsize=(10, 6))
        img = mpimg.imread(gate_plot_path)
        plt.imshow(img)
        plt.axis('off')
        plt.title("Phân phối trọng số Gating (Trục X > 0.5 ưu tiên PTM, < 0.5 ưu tiên Handcrafted)")
        plt.show()
        
    # In thông tin thống kê tóm tắt
    ptm_wins = np.sum(gates > 0.5)
    hc_wins = np.sum(gates <= 0.5)
    print(f"Thống kê nhanh:")
    print(f"   - Số lần ưu tiên PTM (WavLM/HuBERT): {ptm_wins} ({100*ptm_wins/len(gates):.2f}%)")
    print(f"   - Số lần ưu tiên Handcrafted (MFCC/Pitch): {hc_wins} ({100*hc_wins/len(gates):.2f}%)")
else:
    print("ℹChế độ hiện tại không sử dụng Gating. Bỏ qua bước phân tích này.")

## 6. Experiment Comparison

In [ ]:
import pandas as pd

# List all experiments
exp_base_dir = "./outputs/experiments"
if os.path.exists(exp_base_dir):
    experiments = []
    for exp_name_dir in sorted(os.listdir(exp_base_dir)):
        exp_path = os.path.join(exp_base_dir, exp_name_dir)
        results_file = os.path.join(exp_path, "results.json")
        if os.path.exists(results_file):
            with open(results_file, "r") as f:
                data = json.load(f)
                config = data.get("config", {})
                experiments.append({
                    "Experiment": exp_name_dir,
                    "Mode": config.get("mode", ""),
                    "Fusion": config.get("fusion_method", "N/A"),
                    "Feature": config.get("feature_mode", "N/A"),
                    "Best Val Loss": f"{data.get('best_val_loss', 0):.4f}",
                    "Epochs": data.get("epochs_trained", 0),
                })
    
    if experiments:
        df = pd.DataFrame(experiments)
        print("\n" + "="*120)
        print("EXPERIMENT COMPARISON")
        print("="*120)
        print(df.to_string(index=False))
        print("="*120)
    else:
        print("No experiments found.")
else:
    print(f"Directory {exp_base_dir} does not exist yet.")